# 🎬 Multimodal Video Search Engine

**An end-to-end semantic search system for video content using audio transcription, visual understanding, and text embeddings.**

---

## Overview

This project implements a **multimodal search engine** that indexes YouTube videos and enables natural-language search over both spoken content and visual context. Key components:

| Component | Technology | Purpose |
|-----------|------------|---------|
| **Audio → Text** | OpenAI Whisper (base) | Transcribe speech with timestamps |
| **Visual Embeddings** | CLIP (ViT-B/32) | Encode frames into semantic vectors |
| **Text Embeddings** | Sentence-Transformers (all-MiniLM-L6-v2) | Encode transcript for semantic search |
| **Index & Search** | FAISS (CPU) | Fast similarity search over multimodal embeddings |



---
## 1. Install Required Dependencies

We need several libraries:
- **yt-dlp**: Download YouTube videos
- **ffmpeg**: Trim video and extract audio
- **openai-whisper**: Transcribe audio
- **faiss-cpu**: Fast similarity search
- **sentence-transformers**: Text embeddings
- **transformers & torch**: CLIP for image embeddings

In [ ]:

!pip install -q yt-dlp openai-whisper faiss-cpu sentence-transformers transformers torch torchvision

!ffmpeg -version

ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
built with clang version 19.1.7
configuration: --prefix=/opt/anaconda3/envs/ffmpeg_env --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1769712787622/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib --enable-libvo

---
## 2. Imports and Configuration

Import dependencies and set paths. 

In [1]:
import os
import numpy as np
import subprocess
from pathlib import Path


# Configuration 
VIDEO_URL = "https://www.youtube.com/watch?v=ErnWZxJovaM"  # MIT 6.S191 Intro to Deep Learning
CLIP_DURATION_SEC = 300   # 5 minutes for demo
FRAMES_PER_SECOND = 1     # 1 frame/sec → ~300 frames for 5 min

# Paths: use /content for Colab, home dir for local
WORK_DIR = Path("/content/video_search") if os.path.exists("/content") else Path.home() / "video_search"
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

SKIP_DOWNLOAD = False  # Set True by Section 2b if user uploads a file
RAW_VIDEO = WORK_DIR / "raw_video.mp4"
TRIMMED_VIDEO = WORK_DIR / "trimmed_video.mp4"
AUDIO_WAV = WORK_DIR / "audio_16k.wav"
FRAMES_DIR = WORK_DIR / "frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {WORK_DIR}")


Working directory: /Users/nimish/video_search


---
## 2b. Video Source 

Use the default demo video, paste a YouTube URL, or upload your own file. Run this cell *before* Section 3 if you want to use a different video.

In [ ]:
print("Video source:")
print("  1. Default (MIT 6.S191 intro) [Enter]")
print("  2. Paste YouTube URL")
print("  3. Upload video file (Colab only)")
choice = input("Choice [1]: ").strip() or "1"

if choice == "2":
    url = input("Paste YouTube URL: ").strip()
    if url:
        VIDEO_URL = url
        print(f"Using: {VIDEO_URL}")
elif choice == "3":
    try:
        from google.colab import files
        print("Upload your video:")
        uploaded = files.upload()
        for fn in uploaded:
            p = WORK_DIR / fn
            with open(p, "wb") as f:
                f.write(uploaded[fn])
            RAW_VIDEO = p
            SKIP_DOWNLOAD = True
            print(f"Saved to {RAW_VIDEO}")
            break
    except ImportError:
        print("Upload requires Colab. Use option 2 (YouTube URL) instead.")
else:
    print("Using default video.")

---
## 3. Download YouTube Video

Using **yt-dlp** to download the video. We fetch only the needed format to save bandwidth and disk space.

In [2]:
if not SKIP_DOWNLOAD:
    !yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best" -o "{RAW_VIDEO}" "{VIDEO_URL}"
    print("Download complete!")
else:
    print("Using uploaded video (skipped download).")

[youtube] Extracting URL: https://www.youtube.com/watch?v=ErnWZxJovaM
[youtube] ErnWZxJovaM: Downloading webpage
[youtube] ErnWZxJovaM: Downloading android vr player API JSON
[info] ErnWZxJovaM: Downloading 1 format(s): 401+140
[download] /Users/nimish/video_search/raw_video.mp4 has already been downloaded
Download complete!


---


Using **ffmpeg** to cut only the first 5 minutes.

In [3]:
subprocess.run([
    "ffmpeg", "-y", "-i", str(RAW_VIDEO),
    "-t", str(CLIP_DURATION_SEC),  # Duration in seconds
    "-c", "copy",  # No re-encoding = fast
    str(TRIMMED_VIDEO)
], check=True, capture_output=True)

print(f"Trimmed video saved to {TRIMMED_VIDEO}")

Trimmed video saved to /Users/nimish/video_search/trimmed_video.mp4


---
## 5. Extract Audio as 16kHz Mono WAV

Whisper works best with **16kHz mono** audio. ffmpeg converts the video to this format.

In [4]:
subprocess.run([
    "ffmpeg", "-y", "-i", str(TRIMMED_VIDEO),
    "-ac", "1",           # Mono
    "-ar", "16000",       # 16kHz sample rate
    str(AUDIO_WAV)
], check=True, capture_output=True)

print(f"Audio extracted to {AUDIO_WAV}")

Audio extracted to /Users/nimish/video_search/audio_16k.wav


---
## 6. Transcribe Audio with Whisper

 It returns segments with start/end timestamps.

In [5]:
import whisper

model = whisper.load_model("base")
result = model.transcribe(str(AUDIO_WAV), word_timestamps=False)

# result["segments"] = list of {start, end, text}
segments = result["segments"]
print(f"Found {len(segments)} transcript segments")
for i, seg in enumerate(segments[:5]):
    print(f"  {i+1}. [{seg['start']:.1f}s - {seg['end']:.1f}s] {seg['text'][:60]}...")

/opt/anaconda3/envs/ffmpeg_env/lib/python3.11/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Found 57 transcript segments
  1. [0.0s - 15.3s]  Good afternoon, everyone, and welcome to MIT Success 1-9-1....
  2. [15.3s - 19.1s]  My name is Alexander Amini, and I'll be one of your instruc...
  3. [19.1s - 21.4s]  year, along with Ava....
  4. [21.4s - 25.3s]  And together we're really excited to welcome you to this re...
  5. [25.3s - 33.0s]  This is a very fast-paced and very intense one week that we...


---
## 7. Extract Video Frames (1 per second)

We extract one frame every second. Each frame filename includes its timestamp.

In [6]:
subprocess.run([
    "ffmpeg", "-y", "-i", str(TRIMMED_VIDEO),
    "-vf", f"fps={FRAMES_PER_SECOND}",  # 1 frame per second
    str(FRAMES_DIR / "frame_%04d.jpg")
], check=True, capture_output=True)

frame_files = sorted(FRAMES_DIR.glob("frame_*.jpg"))
print(f"Extracted {len(frame_files)} frames")

Extracted 300 frames


---
## 8. Generate Image Embeddings with CLIP

**CLIP (ViT-B/32)** converts each frame into a 512-dimensional vector. These capture visual content.

In [7]:
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = clip_model.to(device).eval()

# Extract frame timestamps from filenames (frame_0001.jpg = 1 sec, etc.)
frame_timestamps = []
frame_embeddings = []

for i, path in enumerate(frame_files):
    ts = float(i)  # 0, 1, 2, ... seconds
    frame_timestamps.append(ts)
    img = Image.open(path).convert("RGB")
    inputs = clip_processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)
    # Newer transformers returns BaseModelOutputWithPooling; extract tensor
    if hasattr(emb, 'last_hidden_state'):
        emb = emb.last_hidden_state[:, 0, :]  # CLS token
    elif hasattr(emb, 'pooler_output') and emb.pooler_output is not None:
        emb = emb.pooler_output
    frame_embeddings.append(emb.cpu().numpy().flatten())

frame_embeddings = np.array(frame_embeddings)
print(f"Image embeddings shape: {frame_embeddings.shape}")

/opt/anaconda3/envs/ffmpeg_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 2376.11it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and

Image embeddings shape: (300, 768)


---
## 9. Generate Text Embeddings with Sentence-Transformers

We use **all-MiniLM-L6-v2** (384 dims) to embed transcript text. Each segment gets a vector that captures its meaning.

In [8]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed each segment's text
segment_texts = [seg["text"].strip() for seg in segments if seg["text"].strip()]
segment_starts = [seg["start"] for seg in segments if seg["text"].strip()]
segment_ends = [seg["end"] for seg in segments if seg["text"].strip()]

# Re-filter segments to match (in case some were empty)
segments_filtered = [s for s in segments if s["text"].strip()]

text_embeddings = text_model.encode(segment_texts)
print(f"Text embeddings shape: {text_embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1074.65it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text embeddings shape: (57, 384)


---
## 10. Align Transcript Segments with Frames by Timestamp

For each transcript segment, we find the frame whose timestamp falls within [start, end] and pair them. We create one combined record per segment.

In [9]:
# For each segment, find the frame closest to segment midpoint
def get_frame_idx_for_segment(seg_start, seg_end, frame_timestamps):
    mid = (seg_start + seg_end) / 2
    idx = np.argmin(np.abs(np.array(frame_timestamps) - mid))
    return idx

aligned_records = []
for i, seg in enumerate(segments_filtered):
    start, end = seg["start"], seg["end"]
    text = seg["text"].strip()
    frame_idx = get_frame_idx_for_segment(start, end, frame_timestamps)
    # Get text embedding (index i) and image embedding (frame_idx)
    text_emb = text_embeddings[i]
    img_emb = frame_embeddings[frame_idx]
    aligned_records.append({
        "start": start,
        "end": end,
        "text": text,
        "text_emb": text_emb,
        "img_emb": img_emb
    })

print(f"Aligned {len(aligned_records)} segment-frame pairs")

Aligned 57 segment-frame pairs


---
## 11. Create Combined Multimodal Embeddings & FAISS Index

We **concatenate** text (384) + image (512) = 896-dim vector per segment. Normalize for cosine similarity, then build FAISS index (CPU).

In [10]:
import faiss

# Concatenate text + image embeddings
combined = np.array([np.concatenate([r["text_emb"], r["img_emb"]]) for r in aligned_records])
combined = combined.astype(np.float32)

# L2 normalize for cosine similarity (inner product = cosine when normalized)
faiss.normalize_L2(combined)

# Build FAISS index (CPU, inner product = cosine for normalized vectors)
dim = combined.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(combined)

print(f"FAISS index built with {index.ntotal} vectors, dim={dim}")

FAISS index built with 57 vectors, dim=1152


---
## 12. Search Demo: Run 4 Curated Queries

Demonstrates semantic search with queries aligned to the video content. Uses `display_results()` for consistent formatting.

In [11]:
def search_video(query, top_k=3):
    """Run a text query and return top-k matching segments."""
    query_text_emb = text_model.encode([query])[0]
    img_emb_dim = len(aligned_records[0]["img_emb"])
    query_img_emb = np.zeros(img_emb_dim, dtype=np.float32)
    query_combined = np.concatenate([query_text_emb, query_img_emb]).astype(np.float32)
    query_combined = np.expand_dims(query_combined, axis=0)
    faiss.normalize_L2(query_combined)
    scores, indices = index.search(query_combined, top_k)
    return [{"start": aligned_records[i]["start"], "end": aligned_records[i]["end"], 
             "text": aligned_records[i]["text"], "score": s} 
            for i, s in zip(indices[0], scores[0])]

def display_results(results, max_text_len=150):
    """Print search results in a consistent, professional format."""
    for rank, r in enumerate(results, 1):
        text_preview = r["text"][:max_text_len] + ("..." if len(r["text"]) > max_text_len else "")
        print(f"\n  [{rank}] {text_preview}")
        print(f"      ⏰ Time: {r['start']:.1f}s - {r['end']:.1f}s  |  Score: {r['score']:.3f}")

# ==============================================================================
# SEARCH DEMO 1: Course Introduction
# ==============================================================================
print("🔍 SEARCH #1: MIT deep learning course introduction")
print("-" * 80)
query = "MIT deep learning course introduction instructors"
results = search_video(query, top_k=3)
display_results(results)

# ==============================================================================
# SEARCH DEMO 2: AI-Generated Content
# ==============================================================================
print("\n\n" + "=" * 80)
print("🔍 SEARCH #2: AI-generated video and costs")
print("-" * 80)
query = "AI-generated video cost compute expensive"
results = search_video(query, top_k=3)
display_results(results)

# ==============================================================================
# SEARCH DEMO 3: Evolution of AI Technology
# ==============================================================================
print("\n\n" + "=" * 80)
print("🔍 SEARCH #3: Commoditization and accessibility")
print("-" * 80)
query = "deep learning commoditized accessible hyperrealistic media"
results = search_video(query, top_k=3)
display_results(results)

# ==============================================================================
# SEARCH DEMO 4: Multiple Query Comparison
# ==============================================================================
print("\n\n" + "=" * 80)
print("🔍 SEARCH DEMO #4: Comparing multiple related queries")
print("=" * 80)
test_queries = [
    "revolutionary impact of artificial intelligence",
    "viral deep learning video million views",
    "rapid evolution and progress in AI field"
]
for idx, q in enumerate(test_queries, 1):
    print("\n" + "-" * 80)
    print(f"Query {idx}/3: \"{q}\"")
    print("-" * 80)
    results = search_video(q, top_k=2)
    display_results(results)

🔍 SEARCH #1: MIT deep learning course introduction
--------------------------------------------------------------------------------

  [1] And welcome to MIT 6S191, the official introductory course on deep learning taught here at MIT.
      ⏰ Time: 123.0s - 134.7s  |  Score: 0.094

  [2] Now fast forward today, the progress in deep learning and people were making all kinds
      ⏰ Time: 236.5s - 241.8s  |  Score: 0.076

  [3] Now over the past decade, in fact, even before we started teaching this course, AI and deep
      ⏰ Time: 45.8s - 52.3s  |  Score: 0.072


🔍 SEARCH #2: AI-generated video and costs
--------------------------------------------------------------------------------

  [1] video took us about $10,000 in compute to generate just about a minute long video.
      ⏰ Time: 215.4s - 221.7s  |  Score: 0.120

  [2] Extremely expensive to compute something we look at like that.
      ⏰ Time: 221.7s - 227.8s  |  Score: 0.081

  [3] of the amazing things that AI and deep learning

---
## 13. Interactive Search

Enter any text query to search the indexed video. Re-run this cell to try different queries.

In [ ]:
# Uses search_video() from the cell above — run Section 12 first
user_query = input("Enter your search query: ").strip() or "AI generated video"

results = search_video(user_query, top_k=3)
print("=" * 70)
print(f"🔍 Query: \"{user_query}\"")
print("=" * 70)
for rank, r in enumerate(results, 1):
    print(f"\n  [{rank}] {r['start']:.1f}s – {r['end']:.1f}s  (score: {r['score']:.3f})")
    print(f"      {r['text']}")